# Task 2a: Token-Wise Function With `torch.nn.RNN`

The dataset maps every input token `x_t` to the target `2 * x_t + K` at the same time step.

## TODO

Implement `RNNModel` using only `torch.nn` modules.

- Use `nn.Embedding` for integer inputs.
- Use `nn.RNN(..., batch_first=True)` even though this task can be solved without recurrence.
- Use `nn.Linear` to produce one classification distribution per time step.
- Set `output_vocab` large enough for the largest target: `2 * 9 + K`.
- Return logits shaped `(B, T, output_vocab)` and do not apply `softmax`.

In [1]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
set_seed(217)
K = 5
device

'cpu'

In [2]:
class CustomDataset(Dataset):
    def __init__(self, vocab_size=10, seq_len=25, size=40000, K=5):
        self.vocab_size = vocab_size
        self.seq_len = seq_len
        self.size = size
        self.X = torch.randint(0, vocab_size, (size, seq_len))
        self.Y = 2 * self.X + K

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

## Model TODO

Use `nn.RNN`, not a hand-written recurrent loop. Think about the vocabulary sizes before coding.

In [3]:
class RNNModel(nn.Module):
    def __init__(self, K, d_model=64, num_layers=2):
        super().__init__()

        # Digits 0-9 are possible inputs
        self.input_vocab = 10

        # Maximum possible cumulative sum + 1
        self.output_vocab = 2 * (self.input_vocab - 1) + K + 1

        # Convert each digit into a d_model-dimensional vector
        self.embedding = nn.Embedding(self.input_vocab, d_model)

        # Process the sequence and maintain a hidden state
        self.rnn = nn.RNN(
            input_size=d_model,
            hidden_size=d_model,
            num_layers=num_layers,
            batch_first=True
        )

        # Convert the hidden state at each time step into output logits
        self.fc = nn.Linear(d_model, self.output_vocab)

    def forward(self, x):
        # x has shape (B, T)
        # B = batch size, T = sequence length

        # Convert tokens to embeddings
        # Shape: (B, T) -> (B, T, d_model)
        x = self.embedding(x)

        # Run the sequence through the RNN
        # out shape: (B, T, d_model)
        out, _ = self.rnn(x)

        # Project each hidden state to output vocabulary size
        # Shape: (B, T, d_model) -> (B, T, output_vocab)
        logits = self.fc(out)

        # Return logits for all time steps
        return logits

In [4]:
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=-1)
            correct += (preds == y).sum().item()
            total += y.numel()
    return correct / total

def train(K=5, epochs=10, batch_size=256):
    model = RNNModel(K=K).to(device)
    dataset = CustomDataset(K=K)
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_set, test_set = random_split(dataset, [train_size, test_size])
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=64)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f'Epoch {epoch + 1}: loss={total_loss / len(train_loader):.4f}, acc={evaluate(model, test_loader, device) * 100:.2f}%')
    return model

# Run after completing RNNModel:
model = train(K=K)

Epoch 1: loss=0.0923, acc=100.00%
Epoch 2: loss=0.0002, acc=100.00%
Epoch 3: loss=0.0001, acc=100.00%
Epoch 4: loss=0.0001, acc=100.00%
Epoch 5: loss=0.0001, acc=100.00%
Epoch 6: loss=0.0000, acc=100.00%
Epoch 7: loss=0.0000, acc=100.00%
Epoch 8: loss=0.0000, acc=100.00%
Epoch 9: loss=0.0000, acc=100.00%
Epoch 10: loss=0.0000, acc=100.00%
